Generar lista de lenguas + códigos

In [2]:
from bs4 import BeautifulSoup
import csv

# Carga el HTML descargado localmente
with open("south_america.txt", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, 'html.parser')
output = []

# Encuentra todas las familias dentro de South America
accordion_groups = soup.select('div.accordion-group')

for group in accordion_groups:
    # Extrae el nombre de la familia
    family_tag = group.select_one('.accordion-heading a')
    if not family_tag:
        continue
    family_name = family_tag.get_text(strip=True).split('(')[0].strip()

    # Encuentra todas las lenguas dentro de esa familia
    language_links = group.select('.accordion-inner a[href^="/languages/language/"]')

    for lang_tag in language_links:
        lang_name = lang_tag.get_text(strip=True)
        href = lang_tag.get('href')
        lang_id = href.split('/')[-1]
        output.append((family_name, lang_name, lang_id))

# Guardar como CSV
with open("south_american_languages.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Family", "Language", "ID"])
    writer.writerows(output)

print("✅ Datos extraídos y guardados en south_american_languages.csv")


✅ Datos extraídos y guardados en south_american_languages.csv


Descargar todas las páginas por ID

In [3]:
import os
import time
import requests

output_dir = 'pages'
base_url = "https://huntergatherer.la.utexas.edu/languages/language/{}"  # ID

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

ids = []
with open('south_american_languages.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        ids.append(row['ID'].strip())

for lang_id in ids:
    url = base_url.format(lang_id)
    dest = os.path.join(output_dir, f"{lang_id}.html")

    if os.path.exists(dest):
        print(f"✔️ Ya descargado: {lang_id}")
        continue

    try:
        print(f"⬇️ Descargando {url}")
        response = requests.get(url)
        response.raise_for_status()
        with open(dest, 'w', encoding='utf-8') as f:
            f.write(response.text)
        time.sleep(1)  # Para evitar sobrecargar el servidor
    except Exception as e:
        print(f"❌ Error al descargar {lang_id}: {e}")


⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/265
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/48
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/257
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/50
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/406
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/51
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/474
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/52
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/53
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/253
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/408
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/343
⬇️ Descargando https://huntergatherer.la.utexas.edu/languages/language/410
⬇️ Descargando https://hunterg

In [8]:
import os
import csv
import time
import traceback
from bs4 import BeautifulSoup

INPUT_CSV = "south_american_languages.csv"
OUTPUT_CSV = "basic_vocabulary_full.csv"
HTML_DIR = "pages"  # Directorio donde están los HTML descargados

COLUMNS = [
    "Gloss Link", "English Gloss", "Spanish Gloss", "Portuguese Gloss", "Orthographic Form", "Phonemicized Form", "Gloss as in Source",
    "Etymology Code", "Proto-Form", "Proto-Language", "Loan Source",
    "Wanderwort Status", "Etyma Set", "Etymology Notes", "General Notes", "Source"
]

def get_basic_vocab_from_file(lang_id):
    filepath = os.path.join(HTML_DIR, f"{lang_id}.html")

    if not os.path.exists(filepath):
        print(f"❌ Archivo no encontrado para ID {lang_id}")
        return []

    with open(filepath, 'r', encoding='utf-8') as f:
        html = f.read()

    soup = BeautifulSoup(html, 'html.parser')

    basic_section = soup.find('div', id='basic_frame')
    if not basic_section:
        print(f"⚠️ Advertencia: No se encontró la sección básica para ID {lang_id}")
        return []

    table = basic_section.find('table')
    if not table:
        print(f"⚠️ Advertencia: No se encontró la tabla de vocabulario para ID {lang_id}")
        return []

    rows = table.select('tbody tr')
    if not rows:
        print(f"⚠️ Advertencia: Tabla vacía para ID {lang_id}")
        return []

    data_rows = []
    for row in rows:
        cells = row.find_all('td')
        # We now expect more columns due to separate glosses, let's adjust the check
        if len(cells) < 14: # Minimum expected cells based on other columns
            print(f"⚠️ Fila ignorada con menos de {len(COLUMNS) - 3} celdas (excluding glosses) en ID {lang_id}")
            continue

        # Extract Gloss Link from the first cell
        a_tag = cells[0].find('a')
        gloss_link = a_tag['href'] if a_tag else ''

        # Extract different language glosses from the first cell
        english_gloss = ""
        spanish_gloss = ""
        portuguese_gloss = ""

        # Assuming glosses are within specific tags or identifiable patterns within the first <td>
        # This part might need adjustment based on actual HTML structure
        # Let's try to find spans with potential language indicators or just extract all text and try to split
        gloss_text = cells[0].get_text(separator=' | ', strip=True)
        # A simple split by ' | ' might not be robust, but let's try to see the structure first
        glosses = [g.strip() for g in gloss_text.split('|')]

        # This is a basic attempt. We might need more sophisticated parsing
        if len(glosses) > 0:
            english_gloss = glosses[0]
        if len(glosses) > 1:
            spanish_gloss = glosses[1]
        if len(glosses) > 2:
            portuguese_gloss = glosses[2]


        # Extract other columns - adjust starting index and count based on the new gloss columns
        # We now have Gloss Link, English, Spanish, Portuguese Glosses, so the next column starts from index 4
        others = [cell.get_text(strip=True) for cell in cells[1:14]] # Still extracting up to 14 cells for now

        # Combine all extracted data
        data_rows.append([gloss_link, english_gloss, spanish_gloss, portuguese_gloss] + others)

    print(f"✅ {len(data_rows)} entradas extraídas para ID {lang_id}")
    return data_rows

def main():
    all_data = []

    with open(INPUT_CSV, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            family = row['Family']
            language = row['Language']
            lang_id = row['ID']

            try:
                print(f"📄 Procesando {language} (ID: {lang_id})...")
                entries = get_basic_vocab_from_file(lang_id)
                if entries:
                    for entry in entries:
                        full_row = [family, language, lang_id] + entry
                        all_data.append(full_row)
                else:
                    print(f"⚠️ No se encontraron entradas para {language} (ID {lang_id})")
            except Exception as e:
                print(f"❌ Error procesando {language} ({lang_id}): {e}")
                traceback.print_exc()

    if all_data:
        with open(OUTPUT_CSV, "w", newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["Family", "Language", "ID"] + COLUMNS)
            writer.writerows(all_data)
        print(f"✅ Datos guardados en {OUTPUT_CSV}")
    else:
        print("⚠️ No se encontraron datos para guardar.")

if __name__ == "__main__":
    main()

📄 Procesando Mapudungun (ID: 265)...
✅ 191 entradas extraídas para ID 265
📄 Procesando Achagua (ID: 48)...
✅ 191 entradas extraídas para ID 48
📄 Procesando Añun (ID: 257)...
⚠️ Advertencia: No se encontró la sección básica para ID 257
⚠️ No se encontraron entradas para Añun (ID 257)
📄 Procesando Apurinã (ID: 50)...
⚠️ Advertencia: No se encontró la sección básica para ID 50
⚠️ No se encontraron entradas para Apurinã (ID 50)
📄 Procesando Asháninka (ID: 406)...
⚠️ Advertencia: No se encontró la sección básica para ID 406
⚠️ No se encontraron entradas para Asháninka (ID 406)
📄 Procesando Asheninka Apurucayali (ID: 51)...
⚠️ Advertencia: No se encontró la sección básica para ID 51
⚠️ No se encontraron entradas para Asheninka Apurucayali (ID 51)
📄 Procesando Bahuana (ID: 474)...
⚠️ Advertencia: No se encontró la sección básica para ID 474
⚠️ No se encontraron entradas para Bahuana (ID 474)
📄 Procesando Baniwa (ID: 52)...
✅ 192 entradas extraídas para ID 52
📄 Procesando Bare (ID: 53)...
✅ 19

In [5]:
import pandas as pd

output_csv_path = "basic_vocabulary_full.csv"

try:
    df = pd.read_csv(output_csv_path)
    print("Column names:")
    print(df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{output_csv_path}' was not found. Please make sure the previous step completed successfully.")
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")

Column names:
['Family', 'Language', 'ID', 'Gloss Link', 'Gloss', 'Orthographic Form', 'Phonemicized Form', 'Gloss as in Source', 'Etymology Code', 'Proto-Form', 'Proto-Language', 'Loan Source', 'Wanderwort Status', 'Etyma Set', 'Etymology Notes', 'General Notes', 'Source']

First 5 rows:


,Family,Language,ID,Gloss Link,Gloss,Orthographic Form,Phonemicized Form,Gloss as in Source,Etymology Code,Proto-Form,Proto-Language,Loan Source,Wanderwort Status,Etyma Set,Etymology Notes,General Notes,Source
Araucanian,Mapudungun,265,/lexical/feature/352,above,"encima, arriba",acima,location,NaN,NaN,wente,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/355,and,y,e,grammar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/359,ash,ceniza(s),cinzas,environment,NaN,NaN,ṭ͡ʀufken,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/360,at,"en, a","em, a",location,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/361,back,espalda,costas,body,NaN,NaN,furi,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
import pandas as pd

output_csv_path = "basic_vocabulary_full.csv"

try:
    df = pd.read_csv(output_csv_path)
    print("Column names:")
    print(df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{output_csv_path}' was not found. Please make sure the previous step completed successfully.")
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")

Column names:
['Family', 'Language', 'ID', 'Gloss Link', 'All Glosses', 'Orthographic Form', 'Phonemicized Form', 'Gloss as in Source', 'Etymology Code', 'Proto-Form', 'Proto-Language', 'Loan Source', 'Wanderwort Status', 'Etyma Set', 'Etymology Notes', 'General Notes', 'Source']

First 5 rows:


,Family,Language,ID,Gloss Link,All Glosses,Orthographic Form,Phonemicized Form,Gloss as in Source,Etymology Code,Proto-Form,Proto-Language,Loan Source,Wanderwort Status,Etyma Set,Etymology Notes,General Notes,Source
Araucanian,Mapudungun,265,/lexical/feature/352,above,"encima, arriba",acima,location,NaN,NaN,wente,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/355,and,y,e,grammar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/359,ash,ceniza(s),cinzas,environment,NaN,NaN,ṭ͡ʀufken,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/360,at,"en, a","em, a",location,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/361,back,espalda,costas,body,NaN,NaN,furi,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
import pandas as pd

output_csv_path = "basic_vocabulary_full.csv"

try:
    df = pd.read_csv(output_csv_path)
    print("Column names:")
    print(df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{output_csv_path}' was not found. Please make sure the previous step completed successfully.")
except Exception as e:
    print(f"An error occurred while reading the CSV file: {e}")

Column names:
['Family', 'Language', 'ID', 'Gloss Link', 'English Gloss', 'Spanish Gloss', 'Portuguese Gloss', 'Orthographic Form', 'Phonemicized Form', 'Gloss as in Source', 'Etymology Code', 'Proto-Form', 'Proto-Language', 'Loan Source', 'Wanderwort Status', 'Etyma Set', 'Etymology Notes', 'General Notes', 'Source']

First 5 rows:


,Family,Language,ID,Gloss Link,English Gloss,Spanish Gloss,Portuguese Gloss,Orthographic Form,Phonemicized Form,Gloss as in Source,Etymology Code,Proto-Form,Proto-Language,Loan Source,Wanderwort Status,Etyma Set,Etymology Notes,General Notes,Source
Araucanian,Mapudungun,265,/lexical/feature/352,above,NaN,NaN,"encima, arriba",acima,location,NaN,NaN,wente,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/355,and,NaN,NaN,y,e,grammar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/359,ash,NaN,NaN,ceniza(s),cinzas,environment,NaN,NaN,ṭ͡ʀufken,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/360,at,NaN,NaN,"en, a","em, a",location,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Araucanian,Mapudungun,265,/lexical/feature/361,back,NaN,NaN,espalda,costas,body,NaN,NaN,furi,NaN,NaN,NaN,NaN,NaN,NaN,NaN
